# B1: Amazon 报告数据处理 | Amazon Report Data Pipeline

> **难度**: ⭐ 入门 Beginner | **预计时间**: 30 分钟 | **前置要求**: Python 基础

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/CBEC-AI-Hub/blob/main/notebooks/b1-data-pipeline.ipynb)

本 Notebook 演示如何用 pandas 处理 Amazon 业务报告数据，包括：
- 读取 Excel 报告文件
- 数据清洗与格式化
- 计算核心业务指标（GMS、Units、ASP）
- 输出汇总报告

This notebook demonstrates how to process Amazon business report data with pandas:
- Reading Excel report files
- Data cleaning and formatting
- Calculating core metrics (GMS, Units, ASP)
- Generating a summary report

---

📎 来源: [CBEC-AI-Hub](https://github.com/kangise/CBEC-AI-Hub) — Path B1 数据采集与处理

In [ ]:
# 安装依赖 | Install dependencies
!pip install pandas openpyxl

## 背景 | Background

跨境电商运营中，Amazon 后台会导出各种业务报告（Business Report、Advertising Report 等），通常是 Excel 或 CSV 格式。

手动处理这些报告既耗时又容易出错。本 Notebook 展示如何用 Python + pandas 自动化这个流程。

In cross-border e-commerce, Amazon provides various business reports (Business Report, Advertising Report, etc.) in Excel/CSV format. Manual processing is time-consuming and error-prone. This notebook shows how to automate the workflow with Python + pandas.

### 核心指标 | Key Metrics

| 指标 Metric | 含义 Meaning | 计算方式 Formula |
|------------|-------------|------------------|
| GMS (Gross Merchandise Sales) | 总销售额 | 单价 × 销量 |
| Units | 销售件数 | 直接统计 |
| ASP (Average Selling Price) | 平均售价 | GMS ÷ Units |

## Step 1: 创建示例数据 | Create Sample Data

我们先创建一份模拟的 Amazon Business Report 数据，模拟真实报告的结构。

We'll create a simulated Amazon Business Report to mimic the real report structure.

In [ ]:
import pandas as pd
import numpy as np

# 创建模拟数据 | Create sample data
np.random.seed(42)

n_rows = 50
categories = ['Electronics', 'Home & Kitchen', 'Sports', 'Beauty', 'Toys']
markets = ['US', 'DE', 'JP']

data = {
    'ASIN': [f'B0{np.random.randint(10000000, 99999999)}' for _ in range(n_rows)],
    'Product Name': [f'Product {i+1} - Sample Item' for i in range(n_rows)],
    'Category': np.random.choice(categories, n_rows),
    'Market': np.random.choice(markets, n_rows),
    'Units Ordered': np.random.randint(0, 500, n_rows),
    'Unit Price': np.round(np.random.uniform(9.99, 199.99, n_rows), 2),
    'Sessions': np.random.randint(50, 5000, n_rows),
    'Page Views': np.random.randint(100, 10000, n_rows),
    'Buy Box %': np.round(np.random.uniform(0.5, 1.0, n_rows), 2),
    'Date': pd.date_range('2025-01-01', periods=n_rows, freq='D').strftime('%Y-%m-%d').tolist()
}

# 添加一些脏数据用于演示清洗 | Add some dirty data for cleaning demo
data['Units Ordered'][3] = -5       # 负数
data['Unit Price'][7] = 0           # 零价格
data['Product Name'][10] = ''       # 空名称
data['Product Name'][15] = None     # None 值

df_raw = pd.DataFrame(data)

# 保存为 Excel | Save as Excel
df_raw.to_excel('sample_business_report.xlsx', index=False)
print(f'✅ 已创建示例报告: sample_business_report.xlsx ({len(df_raw)} 行)')
print(f'✅ Sample report created: sample_business_report.xlsx ({len(df_raw)} rows)')
df_raw.head()

## Step 2: 读取 Excel 文件 | Read Excel File

使用 pandas + openpyxl 读取 Excel 报告。

In [ ]:
# 读取 Excel 文件 | Read Excel file
df = pd.read_excel('sample_business_report.xlsx', engine='openpyxl')

print(f'📊 数据维度 Shape: {df.shape}')
print(f'📊 列名 Columns: {list(df.columns)}')
print()
print('--- 数据类型 Data Types ---')
print(df.dtypes)
print()
print('--- 缺失值 Missing Values ---')
print(df.isnull().sum())

## Step 3: 数据清洗 | Data Cleaning

真实报告中常见的数据问题：
- 空值 / None
- 负数（退货或数据错误）
- 零价格（赠品或错误）

Common data issues in real reports: missing values, negative numbers, zero prices.

In [ ]:
df_clean = df.copy()

# 1. 处理空值 | Handle missing values
missing_before = df_clean.isnull().sum().sum()
df_clean['Product Name'] = df_clean['Product Name'].fillna('Unknown Product')
df_clean['Product Name'] = df_clean['Product Name'].replace('', 'Unknown Product')

# 2. 过滤无效数据 | Filter invalid data
invalid_units = (df_clean['Units Ordered'] < 0).sum()
df_clean = df_clean[df_clean['Units Ordered'] >= 0]

invalid_price = (df_clean['Unit Price'] <= 0).sum()
df_clean = df_clean[df_clean['Unit Price'] > 0]

print(f'🧹 清洗结果 Cleaning Results:')
print(f'   缺失值修复 Missing fixed: {missing_before}')
print(f'   负数订单移除 Negative units removed: {invalid_units}')
print(f'   零价格移除 Zero price removed: {invalid_price}')
print(f'   清洗后行数 Rows after cleaning: {len(df_clean)} (原始 original: {len(df)})')

## Step 4: 计算核心指标 | Calculate Core Metrics

计算每行的 GMS，然后按 Category 和 Market 汇总。

In [ ]:
# 计算 GMS | Calculate GMS
df_clean['GMS'] = df_clean['Units Ordered'] * df_clean['Unit Price']

# 计算转化率 | Calculate conversion rate
df_clean['Conversion Rate'] = np.where(
    df_clean['Sessions'] > 0,
    df_clean['Units Ordered'] / df_clean['Sessions'],
    0
)

print('--- 新增指标预览 New Metrics Preview ---')
df_clean[['ASIN', 'Units Ordered', 'Unit Price', 'GMS', 'Sessions', 'Conversion Rate']].head(10)

## Step 5: 按维度汇总 | Aggregate by Dimensions

In [ ]:
# 按 Category 汇总 | Aggregate by Category
category_summary = df_clean.groupby('Category').agg(
    Total_Units=('Units Ordered', 'sum'),
    Total_GMS=('GMS', 'sum'),
    Avg_Price=('Unit Price', 'mean'),
    Total_Sessions=('Sessions', 'sum'),
    ASIN_Count=('ASIN', 'nunique')
).round(2)

# 计算 ASP | Calculate ASP
category_summary['ASP'] = (category_summary['Total_GMS'] / category_summary['Total_Units']).round(2)

# 排序 | Sort by GMS
category_summary = category_summary.sort_values('Total_GMS', ascending=False)

print('📊 按品类汇总 | Summary by Category')
print('=' * 80)
category_summary

In [ ]:
# 按 Market 汇总 | Aggregate by Market
market_summary = df_clean.groupby('Market').agg(
    Total_Units=('Units Ordered', 'sum'),
    Total_GMS=('GMS', 'sum'),
    Avg_Conversion=('Conversion Rate', 'mean'),
    ASIN_Count=('ASIN', 'nunique')
).round(2)

market_summary['ASP'] = (market_summary['Total_GMS'] / market_summary['Total_Units']).round(2)
market_summary = market_summary.sort_values('Total_GMS', ascending=False)

print('📊 按市场汇总 | Summary by Market')
print('=' * 80)
market_summary

## Step 6: 输出汇总报告 | Generate Summary Report

In [ ]:
# 整体汇总 | Overall summary
total_gms = df_clean['GMS'].sum()
total_units = df_clean['Units Ordered'].sum()
overall_asp = total_gms / total_units if total_units > 0 else 0
avg_conversion = df_clean['Conversion Rate'].mean()

print('=' * 60)
print('📋 业务报告汇总 | Business Report Summary')
print('=' * 60)
print(f'  总 GMS (Total GMS):          ${total_gms:,.2f}')
print(f'  总销量 (Total Units):         {total_units:,}')
print(f'  平均售价 (ASP):               ${overall_asp:.2f}')
print(f'  平均转化率 (Avg CVR):         {avg_conversion:.2%}')
print(f'  ASIN 数量 (Unique ASINs):     {df_clean["ASIN"].nunique()}')
print(f'  覆盖市场 (Markets):           {", ".join(df_clean["Market"].unique())}')
print(f'  数据行数 (Clean Rows):        {len(df_clean)}')
print('=' * 60)

# 导出清洗后的数据 | Export cleaned data
df_clean.to_excel('cleaned_report.xlsx', index=False)
print(f'\n✅ 清洗后数据已导出: cleaned_report.xlsx')
print(f'✅ Cleaned data exported: cleaned_report.xlsx')

## 总结 | Summary

本 Notebook 演示了 Amazon 报告数据处理的基本流程：

1. **读取数据**: 用 `pd.read_excel()` + openpyxl 引擎读取 Excel
2. **数据清洗**: 处理空值、过滤无效数据
3. **指标计算**: GMS = Units × Price, ASP = GMS ÷ Units, CVR = Units ÷ Sessions
4. **多维汇总**: 按 Category、Market 等维度聚合
5. **导出报告**: 输出清洗后的 Excel 文件

### 下一步 | Next Steps

- 📈 [B2: 预测模型与决策](../paths/b-developers/b2-prediction-models.md) — 用 Prophet 做销量预测
- 📖 [Path B 完整路径](../paths/b-developers/README.md) — 查看技术人学习路径
- 🏠 [返回 Hub](https://github.com/kangise/CBEC-AI-Hub)

---

📎 **来源**: [CBEC-AI-Hub](https://github.com/kangise/CBEC-AI-Hub) — AI × 跨境电商实战知识库